In [5]:
from pathlib import Path
import glob
import pandas as pd
import xarray as xr
import cfgrib


def parse_cfsr_filename(filename):
    """
    Parse:
        flxf06.gdas.YYYYMM.grb2

    Returns:
        valid_time
    """

    parts = Path(filename).name.split(".")

    valid_time = pd.to_datetime(
        parts[2],
        format="%Y%m"
    )

    return valid_time


def read_cfsr_variable(file, filter_by_keys, variable_name, output_name=None):

    valid_time = parse_cfsr_filename(file)

    ds = cfgrib.open_dataset(
        file,
        engine="cfgrib",
        filter_by_keys=filter_by_keys,
        decode_timedelta=False,
    )

    # CFS variable name
    da = ds[variable_name]

    # Rename to something cleaner if output_name is provided
    # None is default, I like to keep the original variable name for consistency
    if output_name is not None:
        da = da.rename(output_name)

    da = da.expand_dims(
        valid_time=[valid_time]
    )

    da = da.drop_vars(
        ["time", "step", "heightAboveGround"],
        errors="ignore"    )

    return da

def read_cfsr_2m_temperature(file):
    return read_cfsr_variable(
        file,
        filter_by_keys={
            "typeOfLevel": "heightAboveGround",
            "level": 2,
        },
        variable_name="t2m"
    )

def build_cfsr_temperature_archive(
    input_directory,
    output_zarr,
):
    """
    Build CFSR Zarr archive.

    Dimensions:
        valid_time
        latitude
        longitude
    """

    input_directory = Path(input_directory)

    files = sorted(
        input_directory.glob("flxf06.gdas.*.grb2")
    )

    print(f"Found {len(files)} files.")

    if len(files) == 0:
        raise FileNotFoundError(
            "No CFSR files found."
        )

    monthly = []

    for file in files:

        print(f"Reading {file.name}")

        monthly.append(
            read_cfsr_2m_temperature(
                str(file)
            )
        )

    print("Combining files...")

    da = xr.concat(
        monthly,
        dim="valid_time",
    )

    # Ensure valid_time is sorted
    da = da.sortby("valid_time")

    ds = da.to_dataset()

    lat_size = ds.sizes["latitude"]
    lon_size = ds.sizes["longitude"]

    encoding = {
        "t2m": {
            "chunks": (
                1,
                lat_size,
                lon_size,
            )
        },
        "valid_time": {
            "units": (
                "days since "
                "1970-01-01 00:00:00"
            ),
        },
    }

    ds.to_zarr(
        output_zarr,
        mode="w",
        consolidated=True,
        encoding=encoding,
        zarr_format=2,
    )

    print("Finished.")

In [6]:
build_cfsr_temperature_archive(
    input_directory="/Users/ljob/Desktop/Data/",
    output_zarr="/Users/ljob/Desktop/cnbs-predictor/data/zarr/cfsr_2m_temperature.zarr",
)

Found 2 files.
Reading flxf06.gdas.200610.grb2
Reading flxf06.gdas.200611.grb2
Combining files...
Finished.


In [ ]:
import xarray as xr

zarr_path = "/Users/ljob/Desktop/cnbs-predictor/data/zarr/cfs_2m_temperature.zarr"

ds = xr.open_zarr(
    zarr_path,
    consolidated=True,
)

print(ds)